Digital Business University of Applied Sciences

Data Science und Management (M. Sc.)

DAMI01 / DATA01 Data Analytics

Prof. Dr. Daniel Ambach

Julia Schmid (200022)

***
# Time-Series-Clustering
# Teil 3: Modeling & Evaluation
***


In [7]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import os

from sklearn.preprocessing import StandardScaler
from tslearn.utils import to_time_series_dataset
from sklearn.metrics import adjusted_rand_score

from parameter import INPUT_PATH, OUTPUT_PATH
from funktionen import * 

warnings.simplefilter(action='ignore', category=FutureWarning)

Daten laden

In [15]:
# Zeitreihe laden 
path_temp = INPUT_PATH + "/data_timeseries.csv"
df = pd.read_csv(path_temp)

In [9]:
# Zusatzinfos laden
path_temp = INPUT_PATH + "/data_add_info.csv"
df_add = pd.read_csv(path_temp)

Leeres Evaluations-DatenFrame und Label-Liste initialisieren

In [16]:
df_evaluate = pd.DataFrame(columns=["Durchlauf", "Algorithmus", "Distanzmatrix", "Anzahl_Cluster", "Silhouette_Score", "Davies_Bouldin_Score", "Dunn_Index_Score"])

list_labels = []

***
## Phase 6: Modellierung
***

Im Rahmen der Analyse werden die folgenden Methoden verwendet:

- **Agglomeratives Clustering**: Bildet die hierarchische Struktur natürliche Gruppierungen auf verschiedenen Stufen ab

- **k-Medoids**: Clusterzentren sind echte Zeitreihen aus dem Datensatz sind und keine künstlich berechneten Mittelwerte

- **Fuzzy C-Medoids**: Zeitriehen können mehreren Clustern zugeordnet werden können, wodurch überlappende Muster realistisch abgebildet werden können

- **Spectral Clustering**: Erkennt nicht-konvexe Clusterformen, welche bei Zeitreihen häufig auftreten können

- **DBSCAN**: Clusteranzahl muss vorab nicht festgelegt werden. Zudem werden Ausreißer automatisch identifiziert

Aus dem Grund, dass die Zeitreihen zu groß für die Clusterinmethoden sind, werden fünf Durchläufe mit jeweils 100 Zeitreihen ausgeführt. Bei jedem Durchlauf werden alle Methoden mit der standard DTW, DTW mit Sakoe-Chiba-Band und mit Itakura-Parallelogramm ausgeführ.

In [17]:
X = df.values

n_runs = 5 # Anzahl der Durchläufe
n_sample = 100 # Anzahl der Zeitreihen pro Durchlauf

In [ ]:
all_labels = []

# In jedem der fünf Durchläufen, wird 100 zufällige gewählte Zeitriehen gezogen.
# Jede dieser Zeitreihen wird skaliert und in das tslearn Format gebracht.
# Anschließend werden die DTW-Distanzen bestimmt und auf die Modelle angewendet. 
for i in range(n_runs):

    print(f"Durchlauf #{i+1}:")

    rng = np.random.default_rng(seed=i * 3)
    idx = rng.choice(len(X), n_sample, replace=False)
    X_sample = X[idx]

    # Skalierung
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_sample)

    # tslearn Format
    X_ts = to_time_series_dataset(X_scaled) # tslearn Format 

    # Bestimmte DTWs
    D_dict = getDTW(X_ts)

    # Modelle durchlaufen
    labels_agglomerative = runModel("agglomerative", model_agglomerative, D_dict, df_evaluate, (i+1))
    labels_kmedoids = runModel("kmedoids", model_kmedoids, D_dict, df_evaluate, (i+1))
    labels_fuzzy = runModel("fuzzy", model_fuzzy, D_dict, df_evaluate, (i+1))
    labels_spectral = runModel("spectral", model_spectral, D_dict, df_evaluate, (i+1))
    labels_dbscan = runModel("dbscan", model_dbscan, D_dict, df_evaluate, (i+1), X_scaled)

    run_labels = {"card_id": idx, "Durchlauf": i + 1}
    for dist_name, lbl in labels_agglomerative.items():
        run_labels[f"agglomerative_{dist_name}"] = lbl
    for dist_name, lbl in labels_kmedoids.items():
        run_labels[f"kmedoids_{dist_name}"] = lbl
    for dist_name, lbl in labels_fuzzy.items():
        run_labels[f"fuzzy_{dist_name}"] = lbl
    for dist_name, lbl in labels_spectral.items():
        run_labels[f"spectral_{dist_name}"] = lbl
    for dist_name, lbl in labels_dbscan.items():
        run_labels[f"dbscan_{dist_name}"] = lbl

    all_labels.append(pd.DataFrame(run_labels))

    print("-------------------------------------------------------")

df_labels_all = pd.concat(all_labels, ignore_index=True)


***
## Phase 7: Evaluation
***

### **ARI Stabilität**

In [ ]:
from itertools import combinations

label_cols = [c for c in df_labels_all.columns if c not in ["card_id", "Durchlauf"]]
runs = sorted(df_labels_all["Durchlauf"].unique())

ari_records = []

for col in label_cols:
    for run_a, run_b in combinations(runs, 2):
        df_a = df_labels_all[df_labels_all["Durchlauf"] == run_a][["card_id", col]].dropna()
        df_b = df_labels_all[df_labels_all["Durchlauf"] == run_b][["card_id", col]].dropna()

        # Schnittmenge der card_ids
        merged = df_a.merge(df_b, on="card_id", suffixes=("_a", "_b"))

        if len(merged) < 2:
            continue

        ari = adjusted_rand_score(merged[f"{col}_a"], merged[f"{col}_b"])

        # Algorithmus und Distanzmatrix aus Spaltenname extrahieren
        parts = col.split("_", 1)
        algo = parts[0]
        dist = parts[1] if len(parts) > 1 else "default"

        ari_records.append({
            "Algorithmus": algo,
            "Distanzmatrix": dist,
            "Run_A": run_a,
            "Run_B": run_b,
            "ARI": ari,
            "Overlap": len(merged)
        })

df_ari = pd.DataFrame(ari_records)

df_ari[df_ari["ARI"] < 0.7][["Algorithmus", "Distanzmatrix"]].drop_duplicates()
df_ari.groupby(["Algorithmus", "Distanzmatrix"])["ARI"].mean().reset_index()


### **Evaluationskennzahlen**

In [ ]:
print("Besten Ergebnisse nach Metrik")
sil_row  = df_evaluate.loc[df_evaluate['Silhouette_Score'].idxmax()]
dunn_row = df_evaluate.loc[df_evaluate['Dunn_Index_Score'].idxmax()]
dav_row  = df_evaluate.loc[df_evaluate['Davies_Bouldin_Score'].idxmin()]

print(f"Silhouette Score:    {sil_row['Algorithmus']} {sil_row['Distanzmatrix']} mit {sil_row['Silhouette_Score']:.3f}")
print(f"Dunn Index Score:    {dunn_row['Algorithmus']} {dunn_row['Distanzmatrix']} mit {dunn_row['Dunn_Index_Score']:.3f}")
print(f"Davies Bouldin Score: {dav_row['Algorithmus']} {dav_row['Distanzmatrix']} mit {dav_row['Davies_Bouldin_Score']:.3f}")

In [ ]:
metrics = [
    "Silhouette_Score",
    "Davies_Bouldin_Score",
    "Dunn_Index_Score"
]

algorithms = df_evaluate["Algorithmus"].unique()


for algo in algorithms:
    df_temp = df_evaluate[df_evaluate["Algorithmus"] == algo]

    df_temp = df_temp.melt(
        id_vars=["Durchlauf"],
        value_vars=metrics,
        var_name="Metrik",
        value_name="Score"
    )

    plt.figure(figsize=(8, 4))
    sns.barplot(
        data=df_temp,
        x="Durchlauf",
        y="Score",
        hue="Metrik"
    )

    plt.title(f"Evaluationsmetriken für {algo}")
    plt.xlabel("Durchlauf")
    plt.ylabel("Score")
    plt.legend()
    plt.show()

In [ ]:
#df_mean = df_evaluate.groupby(["Algorithmus", "Distanzmatrix"], as_index=False).mean(numeric_only=True)
df_mean = df_evaluate.groupby(["Algorithmus", "Distanzmatrix"]).agg({
    "Silhouette_Score": "mean",
    "Davies_Bouldin_Score": "mean",
    "Dunn_Index_Score": "mean"
}).reset_index()
df_mean

In [ ]:

metrics = [
    "Silhouette_Score",
    "Davies_Bouldin_Score",
    "Dunn_Index_Score"
]

for metric in metrics:
    plt.figure(figsize=(8, 4))

    sns.barplot(
        data=df_mean,
        x="Algorithmus",
        y=metric,
        hue="Distanzmatrix"
    )

    plt.title(f"{metric} nach Algorithmus und Distanzmatrix")
    plt.xlabel("Algorithmus")
    plt.ylabel(metric)
    plt.legend(title="Distanzmatrix")
    plt.show()

In [ ]:
sdddsdssd

### **Clusteranzahl**

In [13]:
# Labels des ersten Durchlaufs genauer betrachten
df_labels = df_labels_all[df_labels_all["Durchlauf"] == 1].drop(columns=["Durchlauf"])

df_sample_meta = df.iloc[df_labels.index] # Zeitreihen auf Sample selektieren
df_cluster_profile = df.iloc[df_labels["card_id"].values]  # card_id = Zeilenindex in df
df_cluster_profile = df_labels.merge(df_cluster_profile.reset_index(drop=True).assign(card_id=df_labels["card_id"].values), on="card_id")
df_cluster_profile = df_cluster_profile.merge(df_add, how="left", on="card_id")

In [14]:
# # Cluster-Methoden
cluster_methods = [c for c in df_labels.columns if c != "card_id"]

time_cols = []
for col in df_cluster_profile.columns:
    try:
        pd.to_datetime(col)
        time_cols.append(col)
    except Exception:
        continue

# # Zeitindex einmal erstellen
time_index = pd.to_datetime(time_cols)

In [15]:

# Methoden gruppieren
groups = {
    "standard": [m for m in cluster_methods if m.endswith("standard")],
    "sakoe": [m for m in cluster_methods if "sakoe" in m],
    "itakura": [m for m in cluster_methods if "itakura" in m],
}

for group_name, methods in groups.items():
    if not methods:
        continue

    n_methods = len(methods)
    max_clusters = max(len(df_cluster_profile[m].unique()) for m in methods)

    fig = plt.figure(figsize=(5 * max_clusters, 4 * n_methods))
    gs = gridspec.GridSpec(n_methods, max_clusters)

    for row_idx, method in enumerate(methods):
        clusters = sorted(df_cluster_profile[method].unique())
        palette = sns.color_palette("tab10", len(clusters))

        for col_idx, (cluster, color) in enumerate(zip(clusters, palette)):
            ax = fig.add_subplot(gs[row_idx, col_idx])

            df_cluster = df_cluster_profile[df_cluster_profile[method] == cluster]

            # Einzelne Zeitreihen
            for idx, row in df_cluster.iterrows():
                ax.plot(time_index, row[time_cols], color=color, alpha=0.3)

            # Mittelwert + Rolling Mean
            mean_vals = df_cluster[time_cols].mean().values
            mean_series = pd.Series(mean_vals).rolling(window=30, center=True).mean()

            ax.plot(time_index, mean_series, color="black", linewidth=2)

            ax.set_title(f"{method} - Cluster {cluster}")
            ax.tick_params(axis='x', rotation=45)

    plt.suptitle(f"{group_name} – Zeitreihen pro Cluster", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    filepath = os.path.join(OUTPUT_PATH, f"{group_name}_rolling.png")
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close()

In [ ]:
for col in [c for c in cluster_methods]:
    counts = df_labels[col].value_counts().sort_index().reset_index()
    counts.columns = ["Cluster", "Anzahl"]
    counts["Anteil (%)"] = (counts["Anzahl"] / len(df_labels) * 100).round(1)
    print(col)
    print(counts.to_string(index=False))

### **Kontext-Analyse**

In [17]:
df_combined = df_labels.merge(df_add, how='left',  on='card_id') # Zusatzinfos anreichern

In [18]:
numeric_cols = df_combined.select_dtypes(include=np.number).columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in list(cluster_methods) + ["card_id", "client_id"]]

for col in numeric_cols:
    # Long-Format: eine Zeile pro (Beobachtung, Methode)
    df_long = df_combined[cluster_methods + [col]].melt(
        id_vars=[col],
        value_vars=cluster_methods,
        var_name="method",
        value_name="cluster"
    )
    df_long["cluster"] = df_long["cluster"].astype(str)

    all_clusters = sorted(df_long["cluster"].unique(), key=lambda x: int(x) if x.lstrip('-').isdigit() else x)
    palette = sns.color_palette("tab10", len(all_clusters))

    fig, ax = plt.subplots(figsize=(max(14, 2 * len(cluster_methods)), 6))

    sns.boxplot(
        data=df_long,
        x="method",
        y=col,
        hue="cluster",
        palette=palette,
        ax=ax,
        order=cluster_methods,
        hue_order=all_clusters
    )

    ax.set_title(f"Boxplots: {col}", fontsize=14, fontweight='bold')
    ax.set_xlabel("Clustering-Methode")
    ax.set_ylabel(col)
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title="Cluster", bbox_to_anchor=(1.01, 1), loc='upper left')

    plt.tight_layout()
    # Speichern statt anzeigen
    filepath = os.path.join(OUTPUT_PATH, f"Boxplot_{col}.png")
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close()


In [19]:
# cluster_methods = [col for col in df_labels.columns if col != "card_id"]

cat_cols = df_combined.select_dtypes(exclude=np.number).columns.tolist()
cat_cols = [col for col in cat_cols if col not in list(cluster_methods) + ["card_id", "client_id"]]

for col in cat_cols:
    n_methods = len(cluster_methods)

    fig, axes = plt.subplots(n_methods, 2, figsize=(14, 5 * n_methods), squeeze=False)

    for idx, method in enumerate(cluster_methods):
        ct = pd.crosstab(df_combined[method], df_combined[col], normalize='index') * 100

        ax_bar = axes[idx][0]
        ct.plot(kind='bar', stacked=True, ax=ax_bar, colormap='tab10', legend=False)
        ax_bar.set_title(f"{method} – Balken", fontsize=10, fontweight='bold')
        ax_bar.set_xlabel("Cluster")
        ax_bar.set_ylabel("Anteil (%)")
        ax_bar.tick_params(axis='x', rotation=45)

        ax_heat = axes[idx][1]
        sns.heatmap(ct, ax=ax_heat, cmap="YlOrRd", annot=True, fmt=".1f",
                    linewidths=0.5, cbar=True, vmin=0, vmax=100)
        ax_heat.set_title(f"{method} – Heatmap", fontsize=10, fontweight='bold')
        ax_heat.set_xlabel(col)
        ax_heat.set_ylabel("Cluster")
        ax_heat.tick_params(axis='x', rotation=45)
        ax_heat.tick_params(axis='y', rotation=0)

    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels, title=col, loc='lower center',
               bbox_to_anchor=(0.5, -0.01), ncol=min(len(labels), 6), frameon=True)

    plt.suptitle(f"Balkendiagramm & Heatmap: {col}", fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])

    # Speichern 
    filepath = os.path.join(OUTPUT_PATH, f"BarChart_{col}.png")
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close()

### **Signifikantstest**

In [ ]:
from scipy.stats import kruskal, chi2_contingency

results = []
alpha = 0.05
n_tests = len(numeric_cols) + len(cat_cols)
alpha_corrected = alpha / n_tests  # Bonferroni

# Kategoriale Merkmale
for m in cluster_methods:
    for c in cat_cols:
        table_crosstab = pd.crosstab(df_combined[m], df_combined[c])
        chi2, p, dof, expected = chi2_contingency(table_crosstab) # Chi-Quadrat-Kontingenztest
        results.append({
            "Methode": m,
            "Merkmal": c,
            "Test": "Chi-Quadrat",
            "Statistik": round(chi2, 2),
            "p-Wert": round(p, 4),
            "Signifikant": "Ja" if p < alpha_corrected else "Nein"
        })

# Numerische Merkmale
for m in cluster_methods:
    for c in numeric_cols:
        groups = [g[c].dropna().values for _, g in df_combined.groupby(m)]
        if len(groups) >= 2:
            stat, p = kruskal(*groups)
            results.append({
                "Methode": m,
                "Merkmal": c,
                "Test": "Kruskal-Wallis",
                "Statistik": round(stat, 2),
                "p-Wert": round(p, 4),
                "Signifikant": "Ja" if p < alpha_corrected else "Nein"
            })

df_results = pd.DataFrame(results)
df_results

***

In [21]:
# # Alle Variablen und Objekte aus dem Arbeitsspeicher löschen / Speicher freigeben 
%reset -f

***
***